# Module 7b: Distributions of data

**Goal:** describe, identify, fit and **check** the distribution of a variable. In a thesis you must justify which statistical methods you used,
and that justification nearly always starts with "what does the distribution of my data look like?"

**You will learn:** numerical shape measures (skewness, kurtosis) · a catalogue of distributions and the stories behind them · fitting by maximum likelihood ·
comparing models with AIC · checking fit (Q-Q plots, Kolmogorov–Smirnov, Shapiro–Wilk, chi-square goodness-of-fit) · transformations · mixtures

**Maths in this module:** moments · likelihood and MLE · AIC · the KS statistic · χ² goodness-of-fit · Box–Cox · lognormal and Poisson/exponential relationships

> 🎯 Think of a distribution as a **model of how your data were generated**. A good model lets you compute probabilities, simulate, choose tests, and
> explain results. A poor one quietly invalidates your conclusions.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.mixture import GaussianMixture

sns.set_theme(style="whitegrid", palette="colorblind")
desk = pd.read_csv(DATA / "service_desk.csv")
students = pd.read_csv(DATA / "students.csv")
desk.head()

The `service_desk.csv` file records 480 hours at a help desk: `tickets` (counts), `response_minutes` (a time), `satisfaction_pct` (a score capped at 100) and `call_minutes` (a call length). These four variables
deliberately have four *different* shapes.

## 1. Describing shape with numbers

Centre and spread aren't enough. Two further numbers describe shape. With the **central moments** $m_k=\frac1n\sum(x_i-\bar x)^k$:

$$\text{skewness } g_1=\frac{m_3}{m_2^{3/2}}\qquad\qquad \text{excess kurtosis } g_2=\frac{m_4}{m_2^{2}}-3$$

* **Skewness** measures asymmetry: $g_1>0$ means a longer right tail (mean > median); $g_1<0$ a longer left tail; $g_1\approx0$ symmetric.
* **Excess kurtosis** measures tail weight relative to a normal curve (which has $g_2=0$): positive means heavier tails and more extreme values; negative means lighter tails (flat or bimodal shapes).

In [ ]:
def shape_stats(x):
    x = np.asarray(x, dtype=float)
    d = x - x.mean()
    m2, m3, m4 = (d ** 2).mean(), (d ** 3).mean(), (d ** 4).mean()
    return {"n": len(x), "mean": x.mean(), "median": np.median(x), "sd": x.std(ddof=1),
            "skew": m3 / m2 ** 1.5, "excess_kurtosis": m4 / m2 ** 2 - 3}

table = pd.DataFrame({c: shape_stats(desk[c]) for c in desk.columns[1:]}).T.round(2)
table["scipy skew"] = [round(stats.skew(desk[c]), 2) for c in table.index]
table["scipy kurt"] = [round(stats.kurtosis(desk[c]), 2) for c in table.index]
table

By hand and SciPy agree. Reading the table: `response_minutes` is **strongly right-skewed** (skew ≈ 2, heavy tails); `satisfaction_pct` is **left-skewed** (a ceiling at 100);
`call_minutes` has **negative kurtosis** (flat, a hint of bimodality); `tickets` is nearly symmetric.

Rough guide: $|g_1|<0.5$ approximately symmetric, $0.5$–$1$ moderately skewed, $>1$ strongly skewed. Both measures are **noisy in small samples**, so always look at a plot too.

## 2. Looking at the distribution

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, col in zip(axes, desk.columns[1:]):
    sns.histplot(desk[col], stat="density", bins=25, kde=True, ax=ax)
    xs = np.linspace(desk[col].min(), desk[col].max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, desk[col].mean(), desk[col].std()), color="tab:red", linestyle="--", label="normal, same mean & sd")
    ax.set_title(col)
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

A red dashed **normal curve with the same mean and sd** is a quick reality check. Only `tickets` (discrete) is roughly in line with it. Module 9b shows the calculations behind histograms and KDE curves.

## 3. A catalogue of distributions

Each distribution is a *story* about how data arise. Match the story to your data-generating process.

| distribution | the story | parameters | mean | variance | typical data |
|---|---|---|---|---|---|
| **Bernoulli**($p$) | one yes/no trial | $p$ | $p$ | $p(1-p)$ | converted? passed? |
| **Binomial**($n,p$) | number of successes in $n$ independent trials | $n,p$ | $np$ | $np(1-p)$ | how many of 20 users convert |
| **Poisson**($\lambda$) | count of events in a fixed interval, events independent, constant rate | $\lambda$ | $\lambda$ | $\lambda$ | tickets/hour, typos/page |
| **Uniform**($a,b$) | every value in an interval equally likely | $a,b$ | $\frac{a+b}2$ | $\frac{(b-a)^2}{12}$ | random numbers, rounding error |
| **Normal**($\mu,\sigma$) | sum of many small independent effects | $\mu,\sigma$ | $\mu$ | $\sigma^2$ | heights, measurement noise, test scores |
| **Lognormal**($\mu,\sigma$) | *product* of many small independent effects ($\ln X$ is normal) | $\mu,\sigma$ | $e^{\mu+\sigma^2/2}$ | $(e^{\sigma^2}-1)e^{2\mu+\sigma^2}$ | incomes, response times, sizes |
| **Exponential**($\lambda$) | waiting time between Poisson events; "memoryless" | $\lambda$ | $1/\lambda$ | $1/\lambda^2$ | time between arrivals, lifetimes |
| **Gamma**($k,\theta$) | waiting time until the $k$-th event (sum of exponentials) | $k,\theta$ | $k\theta$ | $k\theta^2$ | rainfall totals, insurance claims |
| **Beta**($\alpha,\beta$) | a proportion between 0 and 1 | $\alpha,\beta$ | $\frac{\alpha}{\alpha+\beta}$ | $\frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$ | conversion rates, scores out of 100% |

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
k = np.arange(0, 16); x = np.linspace(0.001, 12, 400)
axes[0, 0].bar(k, stats.binom.pmf(k, 15, 0.3)); axes[0, 0].set_title("Binomial(15, 0.3)")
axes[0, 1].bar(k, stats.poisson.pmf(k, 4.2)); axes[0, 1].set_title("Poisson(4.2)")
axes[0, 2].plot(x, stats.uniform.pdf(x, 2, 6)); axes[0, 2].set_title("Uniform(2, 8)")
axes[0, 3].plot(x, stats.norm.pdf(x, 6, 1.5)); axes[0, 3].set_title("Normal(6, 1.5)")
axes[1, 0].plot(x, stats.lognorm.pdf(x, 0.6, scale=math.exp(1.2))); axes[1, 0].set_title("Lognormal(1.2, 0.6)")
axes[1, 1].plot(x, stats.expon.pdf(x, scale=1 / 0.5)); axes[1, 1].set_title("Exponential(rate 0.5)")
axes[1, 2].plot(x, stats.gamma.pdf(x, 3, scale=1.2)); axes[1, 2].set_title("Gamma(3, 1.2)")
u = np.linspace(0.001, 0.999, 300)
axes[1, 3].plot(u, stats.beta.pdf(u, 8, 2)); axes[1, 3].set_title("Beta(8, 2)")
plt.tight_layout(); plt.show()

### Counts and waiting times are two views of the same process

If events arrive independently at a constant average rate $\lambda$ per hour, then **the number of events per hour is Poisson($\lambda$)** and **the gaps between events are Exponential(rate $\lambda$)**.
Let's simulate arrivals and see both.

In [ ]:
rng = np.random.default_rng(3)
rate = 4.2                                               # events per hour
gaps = rng.exponential(1 / rate, size=60_000)            # hours between arrivals
arrival_times = np.cumsum(gaps)
counts = np.bincount(np.floor(arrival_times).astype(int))[:-1]      # events in each complete hour

ks = np.arange(0, 13)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(ks, np.bincount(counts, minlength=13)[:13] / len(counts), alpha=0.6, label="simulated hourly counts")
axes[0].plot(ks, stats.poisson.pmf(ks, rate), "o-", color="tab:red", label="Poisson(4.2)"); axes[0].legend(); axes[0].set_xlabel("events per hour")
axes[1].hist(gaps * 60, bins=40, density=True, alpha=0.6, label="simulated gaps (minutes)")
xs = np.linspace(0, 60, 200); axes[1].plot(xs, stats.expon.pdf(xs, scale=60 / rate), color="tab:red", label="Exponential"); axes[1].legend(); axes[1].set_xlabel("minutes between events")
plt.tight_layout(); plt.show()
print(f"P(no ticket in an hour) = {stats.poisson.pmf(0, rate):.4f};  P(gap longer than 30 min) = {math.exp(-rate * 0.5):.4f}")

**Poisson check:** a Poisson variable has **mean = variance**. If your counts' variance is much larger than their mean (**overdispersion**), a Poisson model is too optimistic; consider the negative binomial.

## 4. Fitting a distribution: maximum likelihood

Given data $x_1,\dots,x_n$ and a model with parameter $\theta$, the **likelihood** is the probability (density) of the data as a function of $\theta$: $L(\theta)=\prod f(x_i;\theta)$.
The **maximum-likelihood estimate (MLE)** is the $\theta$ that makes the observed data most probable. In practice we maximise the **log-likelihood** $\ell(\theta)=\sum\ln f(x_i;\theta)$ (sums are easier than products).
For common models the maximisation has a neat closed form:

| model | MLE |
|---|---|
| Normal | $\hat\mu=\bar x,\quad\hat\sigma^2=\frac1n\sum(x_i-\bar x)^2$ |
| Exponential (rate) | $\hat\lambda=1/\bar x$ |
| Poisson | $\hat\lambda=\bar x$ |
| Lognormal | $\hat\mu=\overline{\ln x},\quad\hat\sigma^2=\frac1n\sum(\ln x_i-\hat\mu)^2$ (fit a normal to $\ln x$) |
| Binomial($n$ known) | $\hat p=\bar x/n$ |

In [ ]:
x = desk["response_minutes"].to_numpy()

# by hand
mu_hat, sigma_hat = np.log(x).mean(), np.log(x).std()          # ddof=0 is the MLE
# with SciPy (floc=0 fixes the location at 0, so only shape and scale are estimated)
shape, loc, scale = stats.lognorm.fit(x, floc=0)
print(f"lognormal MLE by hand:  mu={mu_hat:.4f}, sigma={sigma_hat:.4f}")
print(f"lognormal via SciPy:    mu={math.log(scale):.4f}, sigma={shape:.4f}   (scale = e^mu)")
print(f"exponential rate by hand {1 / x.mean():.4f}  |  SciPy scale {stats.expon.fit(x, floc=0)[1]:.3f} (= 1/rate = mean)")

### Comparing candidate models with AIC

A model with more parameters always fits at least as well, so compare models with a penalty for complexity. The **Akaike information criterion** is

$$\text{AIC}=2k-2\ell_{\max}$$

where $k$ is the number of fitted parameters. **Lower is better**; a difference of more than about 10 is strong evidence for the lower one. (BIC replaces $2k$ by $k\ln n$ and penalises complexity more.)

In [ ]:
fits = {
    "normal": (stats.norm, stats.norm.fit(x), 2),
    "lognormal": (stats.lognorm, stats.lognorm.fit(x, floc=0), 2),
    "gamma": (stats.gamma, stats.gamma.fit(x, floc=0), 2),
    "exponential": (stats.expon, stats.expon.fit(x, floc=0), 1),
}
rows = []
for name, (dist, params, k) in fits.items():
    loglik = dist.logpdf(x, *params).sum()
    rows.append((name, k, loglik, 2 * k - 2 * loglik))
aic_table = pd.DataFrame(rows, columns=["model", "k", "log-likelihood", "AIC"]).set_index("model").round(1).sort_values("AIC")
aic_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(x, bins=40, density=True, alpha=0.5, color="grey")
xs = np.linspace(0.01, x.max(), 400)
for name, (dist, params, k) in fits.items():
    ax.plot(xs, dist.pdf(xs, *params), label=f"{name} (AIC {aic_table.loc[name, 'AIC']:.0f})")
ax.set_xlabel("Response minutes"); ax.legend(); ax.set_xlim(0, 40)
plt.show()

The lognormal wins decisively (AIC about 2,820 vs 2,842 for the gamma and 3,061 for the normal), matching the *story*: response times are the product of many small delays.
**AIC says which candidate is best among those you tried, not that the winner is good.** Always check the fit.

## 5. Checking the fit

### Q-Q plots
A **quantile–quantile plot** compares your sorted data with the quantiles a model *predicts*. For the $i$-th smallest of $n$ values, the plotting position is $p_i=(i-0.5)/n$ and the model's quantile is $F^{-1}(p_i)$.
If the model is right, the points fall on a straight line (for the normal: $y=\mu+\sigma x$).

In [ ]:
def qq_points(data, dist=stats.norm):
    data = np.sort(np.asarray(data, dtype=float))
    n = len(data)
    p = (np.arange(1, n + 1) - 0.5) / n
    return dist.ppf(p), data            # theoretical quantiles, sample quantiles

rng = np.random.default_rng(5)
samples = {
    "normal data": rng.normal(0, 1, 300),
    "right-skewed (lognormal)": rng.lognormal(0, 0.8, 300),
    "heavy tails (t, 3 df)": rng.standard_t(3, 300),
    "bimodal": np.r_[rng.normal(-2, 0.7, 150), rng.normal(2, 0.7, 150)],
}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (name, data) in zip(axes, samples.items()):
    th, sm = qq_points(data)
    ax.scatter(th, sm, s=8)
    slope, intercept = np.polyfit(th, sm, 1)
    ax.plot(th, slope * th + intercept, color="tab:red")
    ax.set_title(name); ax.set_xlabel("theoretical normal quantiles"); ax.set_ylabel("sample quantiles")
plt.tight_layout(); plt.show()

**How to read a Q-Q plot against the normal line:**

| pattern | meaning |
|---|---|
| points on the line | good fit |
| curve bending up at the right (convex) | right-skewed |
| S-shape: low tail below the line, high tail above | heavy tails |
| plateaus / steps with a gap | bimodal or discrete |

Now the real data. The response times against a normal model, and against the fitted lognormal (using the same calculation with the lognormal's quantile function):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
th, sm = qq_points(x, stats.norm(x.mean(), x.std()))
axes[0].scatter(th, sm, s=8); axes[0].plot([sm.min(), sm.max()], [sm.min(), sm.max()], color="tab:red"); axes[0].set_title("vs normal: curves away")
th, sm = qq_points(x, stats.lognorm(*stats.lognorm.fit(x, floc=0)))
axes[1].scatter(th, sm, s=8); axes[1].plot([sm.min(), sm.max()], [sm.min(), sm.max()], color="tab:red"); axes[1].set_title("vs fitted lognormal: on the line")
for ax in axes: ax.set_xlabel("theoretical quantiles"); ax.set_ylabel("observed quantiles")
plt.tight_layout(); plt.show()

### Formal tests of fit

**Kolmogorov–Smirnov (KS)** measures the biggest vertical gap between the empirical CDF $F_n$ and the model CDF $F$:

$$D=\max_i\max\Big(\tfrac in-F(x_{(i)}),\;F(x_{(i)})-\tfrac{i-1}n\Big)$$

In [ ]:
def ks_statistic(data, cdf):
    xs = np.sort(np.asarray(data, dtype=float))
    n = len(xs)
    F = cdf(xs)
    i = np.arange(1, n + 1)
    return max((i / n - F).max(), (F - (i - 1) / n).max())

lognormal = stats.lognorm(*stats.lognorm.fit(x, floc=0))
print("KS statistic by hand :", round(ks_statistic(x, lognormal.cdf), 4))
print("scipy.stats.kstest   :", stats.kstest(x, lognormal.cdf))
print("against a normal     :", stats.kstest(x, stats.norm(x.mean(), x.std()).cdf))

> ⚠️ **Caveat:** the standard KS p-value assumes the model's parameters were fixed *in advance*. When they're estimated from the same data (as here), the p-value is too generous
> (the Lilliefors correction fixes this for the normal). Use KS mostly to compare candidates by $D$, and rely on Q-Q plots for the verdict.

**Shapiro–Wilk** is a powerful test of *normality* (best for $n<5000$). Its null hypothesis is "the data are normal":

In [ ]:
print("exam scores           :", stats.shapiro(students["exam_score"]))
print("response minutes      :", stats.shapiro(x))
print("log(response minutes) :", stats.shapiro(np.log(x)))

Exam scores are not rejected as non-normal (p = 0.07, though it's borderline); raw response times are overwhelmingly rejected; **after taking logs, the normality is not rejected** (p = 0.58). Consistent with the lognormal story.

### Tests are not magic: sample size changes everything
A formal normality test with a huge sample flags trivial deviations; with a tiny sample it sees nothing. Take the same *slightly* skewed gamma distribution (skewness ≈ 0.28) at two sample sizes:

In [ ]:
big = np.random.default_rng(0).gamma(50, size=4000)
print(f"skewness of the population-like sample: {stats.skew(big):.2f}")
print(f"n = 50    -> Shapiro p = {stats.shapiro(big[:50]).pvalue:.3f}   (not rejected)")
print(f"n = 4000  -> Shapiro p = {stats.shapiro(big).pvalue:.1e}   (rejected, for a departure that hardly matters)")

**Practical rule for a thesis:** decide with **plots + effect (skewness) + the sensitivity of your method + sample size**, and use the formal test as supporting evidence, not as an on/off switch.
Many methods (t-tests, regression) are robust to mild non-normality when $n$ is moderate (the CLT helps). Strong skewness, outliers, or small $n$ are what matter.

### Goodness-of-fit for counts: the chi-square test
For discrete data compare **observed** counts with **expected** counts under the model: $\chi^2=\sum\frac{(O-E)^2}{E}$, with degrees of freedom $=(\text{number of bins})-1-(\text{parameters estimated})$.
Expected counts should be at least about 5, so merge sparse bins. Testing whether the `tickets` per hour are Poisson:

In [ ]:
tickets = desk["tickets"].to_numpy()
n = len(tickets)
lam = tickets.mean()                                           # MLE of the Poisson rate

observed = np.array([(tickets <= 1).sum()] + [(tickets == k).sum() for k in range(2, 9)] + [(tickets >= 9).sum()])
pm = stats.poisson.pmf(np.arange(0, 9), lam)
expected = n * np.r_[stats.poisson.cdf(1, lam), pm[2:9], stats.poisson.sf(8, lam)]

chi2 = ((observed - expected) ** 2 / expected).sum()
dof = len(observed) - 1 - 1                                    # one parameter (lambda) was estimated
print(pd.DataFrame({"observed": observed, "expected": expected.round(1)}, index=["<=1", 2, 3, 4, 5, 6, 7, 8, ">=9"]).T)
print(f"\nlambda-hat = {lam:.2f}, chi2 = {chi2:.2f}, df = {dof}, p = {stats.chi2.sf(chi2, dof):.2f}")
print(f"mean {tickets.mean():.2f} vs variance {tickets.var(ddof=1):.2f} (Poisson: equal)")

$p=0.66$: nothing contradicts the Poisson model, and the mean ≈ variance check agrees.

## 6. Transformations: reshaping skewed data

A monotone transformation can make a skewed variable more symmetric, which often makes plots readable and methods (t-tests, regression) behave better:

| transform | helps when | notes |
|---|---|---|
| $\ln x$ (log) | right skew, multiplicative effects, positive data | use $\ln(1+x)$ if zeros occur |
| $\sqrt x$ | counts, mild right skew | |
| $1/x$ | very strong right skew | reverses order |
| **Box–Cox** $\frac{x^\lambda-1}{\lambda}$ ($\lambda=0\Rightarrow\ln x$) | choose $\lambda$ by maximum likelihood | positive data only |

In [ ]:
transformed, lam_bc = stats.boxcox(x)
print(f"Box-Cox lambda = {lam_bc:.2f}  (close to 0, so the log transform is the natural choice)")
versions = {"raw": x, "square root": np.sqrt(x), "log": np.log(x), "Box-Cox": transformed}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (name, v) in zip(axes, versions.items()):
    ax.hist(v, bins=30, edgecolor="white"); ax.set_title(f"{name}: skew = {stats.skew(v):.2f}")
plt.tight_layout(); plt.show()

### Reporting on the log scale: differences become ratios

Because $\ln a-\ln b=\ln(a/b)$, a **difference of logs is a ratio**. Compare time spent in the app between two experimental groups, analysing the logs and converting back with $\exp$:

In [ ]:
trial = pd.read_csv(DATA / "app_trial.csv")
log_game = np.log(trial.loc[trial["group"] == "gamified", "minutes_active"])
log_ctrl = np.log(trial.loc[trial["group"] == "control", "minutes_active"])

res = stats.ttest_ind(log_game, log_ctrl, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"Gamified users spent {math.exp(log_game.mean() - log_ctrl.mean()):.2f}x as many minutes as control "
      f"(95% CI {math.exp(ci.low):.2f} to {math.exp(ci.high):.2f}; p = {res.pvalue:.3f}).")

That is a *geometric mean ratio*: a multiplicative effect, the natural way to describe skewed, positive quantities. Always state that you analysed the log scale.

**Alternative to transforming:** use methods that don't assume normality (permutation or bootstrap, or the rank-based tests of Module 8b).

## 7. The distributions of *statistics*: $t$, $\chi^2$ and $F$

Some distributions exist to describe **what a statistic does across repeated samples**. They're the engines of the tests in Module 8b:

* **$t_\nu$:** the standardised sample mean when $\sigma$ is *estimated*: $t=\dfrac{\bar x-\mu}{s/\sqrt n}$ with $\nu=n-1$ degrees of freedom. Heavier tails than the normal (more spread for small $n$).
* **$\chi^2_k$:** the sum of $k$ squared independent standard normals. Used for variances and goodness-of-fit/independence tests.
* **$F_{\nu_1,\nu_2}$:** a ratio of two (scaled) $\chi^2$s, i.e. a ratio of variances. Used in ANOVA and regression.

In [ ]:
rng = np.random.default_rng(1)
z = rng.normal(0, 1, (20_000, 5))                                  # 20,000 samples of size 5 from N(0,1)
t_stat = z.mean(axis=1) / (z.std(axis=1, ddof=1) / np.sqrt(5))
chi_sq = (rng.normal(size=(20_000, 3)) ** 2).sum(axis=1)           # sum of 3 squared standard normals
f_ratio = rng.normal(size=(20_000, 10)).var(axis=1, ddof=1) / rng.normal(size=(20_000, 12)).var(axis=1, ddof=1)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
xs = np.linspace(-6, 6, 300)
axes[0].hist(t_stat, bins=np.linspace(-6, 6, 80), density=True, alpha=0.5)
axes[0].plot(xs, stats.t.pdf(xs, 4), color="tab:red", label="t, 4 df"); axes[0].plot(xs, stats.norm.pdf(xs), "k--", label="normal"); axes[0].legend(); axes[0].set_title("t statistic, n = 5")
xs = np.linspace(0, 15, 300); axes[1].hist(chi_sq, bins=60, density=True, alpha=0.5); axes[1].plot(xs, stats.chi2.pdf(xs, 3), color="tab:red"); axes[1].set_title("chi-square, 3 df")
xs = np.linspace(0, 6, 300); axes[2].hist(f_ratio, bins=np.linspace(0, 6, 80), density=True, alpha=0.5); axes[2].plot(xs, stats.f.pdf(xs, 9, 11), color="tab:red"); axes[2].set_title("F ratio, (9, 11) df")
plt.tight_layout(); plt.show()
print(f"P(|t| > 2.5): simulated {np.mean(np.abs(t_stat) > 2.5):.3f} | t(4 df) {2 * stats.t.sf(2.5, 4):.3f} | normal {2 * stats.norm.sf(2.5):.3f}")

Ignoring the extra uncertainty from estimating $\sigma$ would make a "significant" result five times too easy to get at $n=5$ (0.067 vs 0.012). That's why we use $t$, not $z$, in practice.

## 8. Mixtures: when one distribution isn't enough

`call_minutes` has two bumps. That usually signals **two populations mixed together** (quick calls and complex calls). A mean of 6.6 minutes describes *neither*.
A **Gaussian mixture model** fits a weighted sum of normals; compare the number of components with BIC (lower is better).

In [ ]:
calls = desk[["call_minutes"]].to_numpy()
for k in (1, 2, 3):
    gm = GaussianMixture(n_components=k, random_state=0).fit(calls)
    print(f"{k} component(s): BIC = {gm.bic(calls):7.1f}   weights = {gm.weights_.round(2)}   means = {gm.means_.ravel().round(1)}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(calls, bins=40, density=True, alpha=0.5)
gm2 = GaussianMixture(2, random_state=0).fit(calls)
xs = np.linspace(0, calls.max(), 400).reshape(-1, 1)
ax.plot(xs, np.exp(gm2.score_samples(xs)), color="tab:red", label="2-component mixture")
ax.set_xlabel("Call minutes"); ax.legend(); plt.show()

Two components win clearly: about **61%** short calls (mean ≈ 3 min) and **39%** long calls (mean ≈ 12 min). Report the groups separately (or find the variable that separates them) instead of one misleading average.

## 9. Which distribution? A decision aid

| your variable | think of | check |
|---|---|---|
| yes/no outcome | Bernoulli / Binomial | proportion CIs (Module 8b) |
| count of events per interval | Poisson (or negative binomial if variance > mean) | mean ≈ variance? χ² GOF |
| time between events / lifetime | Exponential, Gamma, Weibull | Q-Q plot, AIC |
| positive, right-skewed (money, times, sizes) | Lognormal, Gamma | log-transform; Q-Q of logs |
| proportion in (0, 1) | Beta | bounded; left/right skew |
| measurement with symmetric noise | Normal | Q-Q, Shapiro, skew ≈ 0 |
| two bumps | mixture | BIC for number of components |
| heavy tails / extremes | $t$, Pareto, extreme-value | Q-Q S-shape; robust methods |

A one-page version is in [`reference/distributions-cheatsheet.md`](../../reference/distributions-cheatsheet.md).

## 10. How to write it up

> *Response times were strongly right-skewed (skewness 2.1; Shapiro–Wilk p < 0.001), so they were analysed on the log scale, where the distribution was approximately normal (skewness −0.1; Shapiro–Wilk p = 0.58; Q-Q plot in Figure 2).
> A lognormal model fitted better than gamma, exponential or normal models (AIC 2,820 vs 2,842, 3,085 and 3,061). Effects are reported as ratios of geometric means.*

Say **what you checked, how, and what you did about it.** Reviewers look for exactly this.

## Recap

* Describe shape with **skewness and kurtosis** (and a plot). Match a distribution to the *data-generating story*.
* **MLE** fits a model; **AIC** compares candidates; **Q-Q plots** and tests check them, but sample size changes what tests flag.
* **Transform** (log, Box–Cox) or use distribution-free methods when data are skewed. On the log scale, differences become ratios.
* $t$, $\chi^2$ and $F$ are distributions of *statistics*, behind the tests in Module 8b.
* Bumps signal mixtures; report subgroups, not a misleading mean.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** (a distribution profile of your own variables).